# 02 - Limpeza: tempos de trânsito com erro de registro

**Objetivo:** encontrar e tratar os tempos de trânsito implausíveis vistos no notebook 01, gerando as bases `interim` e `processed`.

**Fluxo de dados (diretriz do projeto):** `data/raw` (CSV original, nunca alterado) -> `data/interim/entregas_validadas.parquet` (valores originais + marcação) -> `data/processed/entregas_limpas.parquet` (valores tratados).

**Código reutilizável:**
- `src/logistica_rotas/limpeza/tratamento.py` - regra de outlier e limpeza.
- `src/logistica_rotas/metricas/kpis.py` - OTD, atraso, cancelamento e `resumo_operacional`.
- Testes em `tests/` (`.venv\Scripts\python.exe -m pytest`).

## 1. Configuração

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
sys.path.insert(0, str(RAIZ / "src"))

import logistica_rotas.ingestao.simulacao as simulacao
from logistica_rotas.ingestao.leitura import carregar_entregas
from logistica_rotas.limpeza.tratamento import (
    CAMINHO_INTERIM, CAMINHO_PROCESSED, FATOR_OUTLIER, ROTULOS_FAIXAS,
    adicionar_colunas_derivadas, limpar_entregas, razao_tempo_faixa, salvar_parquet, validar_entregas,
)
from logistica_rotas.metricas.kpis import resumo_operacional, taxa_atraso, taxa_otd

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.width", 140)

In [2]:
bruto = carregar_entregas()
base = adicionar_colunas_derivadas(bruto)
concl = base[base["status"] != "cancelado"]
print(f"{len(bruto):,} pedidos | {len(concl):,} concluídos (com tempo)".replace(",", "."))

5.000 pedidos | 4.790 concluídos (com tempo)


## 2. A regra do notebook 01 (velocidade < 3 km/h) não serve

No notebook 01, marcamos como suspeitas as entregas com velocidade média abaixo de 3 km/h. O problema: o tempo de trânsito inclui um **tempo fixo** de coleta e entrega (esperar o pedido, estacionar, subir no prédio). Em entregas curtas, esse tempo fixo pesa muito e derruba a velocidade média, mesmo sem nenhum erro.

In [3]:
lentas = concl["velocidade_media_kmh"] < 3
pd.DataFrame({
    "entregas": concl.groupby("faixa_distancia_km", observed=True).size(),
    "velocidade < 3 km/h": concl[lentas].groupby("faixa_distancia_km", observed=True).size(),
}).fillna(0).astype(int)

,entregas,velocidade < 3 km/h
faixa_distancia_km,,
0-2,1047,41
2-4,1354,9
4-6,863,1
6-8,517,3
8-10,347,2
10-15,440,0
15+,222,0


**Leitura:** se as entregas "lentas" se concentram nas faixas mais curtas, a regra está pegando o efeito do tempo fixo, e não erros.

## 3. Regra proposta: comparar com entregas parecidas

Para cada entrega, dividimos o tempo pela **mediana do tempo das entregas na mesma faixa de distância** (razão). Uma razão 1 é um tempo típico; razão 3 é o triplo do típico. A mediana é usada porque não é puxada pelos próprios outliers.

In [4]:
razao = razao_tempo_faixa(concl)
razao.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame("razão tempo / mediana da faixa")

,razão tempo / mediana da faixa
count,4790.00
mean,1.09
std,0.58
min,0.36
50%,1.00
90%,1.48
99%,2.26
99.9%,9.31
max,11.08


In [5]:
sensibilidade = pd.Series(
    {f"razão > {k}": int((razao > k).sum()) for k in (2.0, 2.5, 3.0, 3.5, 4.0, 5.0)},
    name="entregas marcadas",
)
sensibilidade.to_frame()

,entregas marcadas
razão > 2.0,85
razão > 2.5,38
razão > 3.0,34
razão > 3.5,32
razão > 4.0,29
razão > 5.0,23


**Leitura:** a contagem cai rápido até perto de 3 e depois quase estabiliza - sinal de que acima disso sobra um grupo separado (os erros), e não a cauda natural dos tempos.

## 4. Validação com gabarito

Vantagem de dados simulados: dá para saber **exatamente** quais tempos foram corrompidos. Geramos a mesma base (mesma seed) com a probabilidade de erro zerada; as linhas cujo tempo mudou são os erros propositais. Assim medimos acertos, falsos positivos (entrega boa marcada) e erros perdidos de cada regra.

> Com dados reais não existe gabarito - por isso a validação aqui é tão valiosa para escolher o método.

In [6]:
original = simulacao.PROB_OUTLIER_TEMPO
simulacao.PROB_OUTLIER_TEMPO = 0.0
try:
    sem_erros = simulacao.gerar_entregas()
finally:
    simulacao.PROB_OUTLIER_TEMPO = original

gabarito = (bruto["tempo_transito_min"].to_numpy() != sem_erros["tempo_transito_min"].to_numpy()) \
    & bruto["tempo_transito_min"].notna().to_numpy()
gabarito = pd.Series(gabarito, index=base.index)
razao_total = razao_tempo_faixa(base)

def avaliar(marcados: pd.Series) -> dict:
    marcados = marcados.fillna(False)
    return {
        "marcados": int(marcados.sum()),
        "acertos": int((marcados & gabarito).sum()),
        "falsos positivos": int((marcados & ~gabarito).sum()),
        "erros perdidos": int((~marcados & gabarito).sum()),
    }

print("Erros propositais na base:", int(gabarito.sum()))
pd.DataFrame({
    "velocidade < 3 km/h (notebook 01)": avaliar(base["velocidade_media_kmh"] < 3),
    "razão > 2,5": avaliar(razao_total > 2.5),
    "razão > 3 (escolhida)": avaliar(razao_total > 3),
    "razão > 4": avaliar(razao_total > 4),
}).T

Erros propositais na base: 32


,marcados,acertos,falsos positivos,erros perdidos
velocidade < 3 km/h (notebook 01),56,24,32,8
"razão > 2,5",38,32,6,0
razão > 3 (escolhida),34,32,2,0
razão > 4,29,29,0,3


**Decisão:** razão > 3 (constante `FATOR_OUTLIER` em `tratamento.py`) + velocidade > 60 km/h (fisicamente impossível para moto urbana). Registrada em `docs/dicionario_dados.md`.

**Tratamento:** não apagamos a linha (o pedido existiu e tem cidade, bairro, valor). Só o tempo é inválido: ele fica vazio na base tratada, o valor original é guardado em `tempo_transito_min_bruto` e a coluna `outlier_tempo` marca o caso. O status desses pedidos foi calculado com o tempo errado, então eles também saem do OTD e da taxa de atraso.

## 5. Aplicar e salvar

In [7]:
validada = validar_entregas(bruto)   # interim: valores originais + marcação
limpa = limpar_entregas(validada)      # processed: tempo dos outliers = NaN

salvar_parquet(validada, CAMINHO_INTERIM)
salvar_parquet(limpa, CAMINHO_PROCESSED)
print("Outliers marcados:", int(limpa["outlier_tempo"].sum()))
print("Salvo:", CAMINHO_INTERIM.relative_to(RAIZ))
print("Salvo:", CAMINHO_PROCESSED.relative_to(RAIZ))

Outliers marcados: 34
Salvo: data\interim\entregas_validadas.parquet
Salvo: data\processed\entregas_limpas.parquet


In [8]:
limpa.loc[limpa["outlier_tempo"], ["id_pedido", "local_origem", "local_destino", "distancia_km",
                                     "tempo_transito_min_bruto", "prazo_prometido_min", "status"]].head(10)

,id_pedido,local_origem,local_destino,distancia_km,tempo_transito_min_bruto,prazo_prometido_min,status
552,PED000553,Jardim Zaíra · Mauá,Jardim Itapeva · Mauá,3.83,138.70,25,com_atraso
893,PED000894,Planalto · SBC,Baeta Neves · SBC,3.60,129.60,25,com_atraso
930,PED000931,Casa Branca · SA,Vila Assunção · SA,1.69,42.90,20,com_atraso
957,PED000958,Paulicéia · SBC,Centro · SBC,7.77,149.90,35,com_atraso
962,PED000963,Vila Bocaina · Mauá,Jardim Zaíra · Mauá,3.92,111.80,25,com_atraso
1063,PED001064,Centro · SBC,Baeta Neves · SBC,2.66,61.50,20,com_atraso
1065,PED001066,Rudge Ramos · SBC,Parque São Vicente · Mauá,12.64,185.30,45,com_atraso
1374,PED001375,Centro · SA,Bairro Jardim · SA,1.74,59.80,20,com_atraso
1638,PED001639,Barão de Mauá · Mauá,Parque São Vicente · Mauá,2.34,58.80,20,com_atraso
1779,PED001780,Parque São Vicente · Mauá,Vila Noêmia · Mauá,1.01,52.90,15,com_atraso


## 6. Antes x depois

In [9]:
def indicadores(df: pd.DataFrame) -> pd.Series:
    feitas = df[df["status"] != "cancelado"]
    return pd.Series({
        "OTD %": taxa_otd(df),
        "taxa de atraso %": taxa_atraso(df),
        "tempo médio (min)": feitas["tempo_transito_min"].mean(),
        "tempo mediano (min)": feitas["tempo_transito_min"].median(),
        "desvio padrão do tempo": feitas["tempo_transito_min"].std(),
        "tempo máximo (min)": feitas["tempo_transito_min"].max(),
    })

comparacao = pd.DataFrame({"antes (raw)": indicadores(bruto), "depois (processed)": indicadores(limpa)})
comparacao["diferença"] = comparacao["depois (processed)"] - comparacao["antes (raw)"]
comparacao

,antes (raw),depois (processed),diferença
OTD %,91.90,92.56,0.66
taxa de atraso %,8.10,7.44,-0.66
tempo médio (min),20.28,19.57,-0.71
tempo mediano (min),15.95,15.90,-0.05
desvio padrão do tempo,16.38,12.82,-3.56
tempo máximo (min),297.70,128.90,-168.80


**Leitura:** a média e o desvio padrão mudam bastante; a mediana quase não muda - é exatamente por isso que usamos a mediana nos resumos.

In [10]:
pd.concat({
    "antes": base.groupby("trajeto").apply(resumo_operacional, include_groups=False)["otd_%"],
    "depois": limpa.groupby("trajeto").apply(resumo_operacional, include_groups=False)["otd_%"],
}, axis=1)

,antes,depois
trajeto,,
entre cidades,83.47,84.15
mesma cidade,94.10,94.75


## 7. Conferência da base salva

O Parquet guarda os tipos (data com fuso, categorias, booleanos) - ao contrário do CSV, não precisa converter de novo na leitura.

In [11]:
relida = pd.read_parquet(CAMINHO_PROCESSED)
print("Linhas iguais:", len(relida) == len(bruto))
print("Fuso preservado:", relida["data_hora_pedido"].dt.tz)
relida.dtypes.to_frame("tipo").T

Linhas iguais: True
Fuso preservado: America/Sao_Paulo


,id_pedido,data_hora_pedido,cidade_origem,bairro_origem,cidade_destino,bairro_destino,endereco_destino,lat_origem,lon_origem,lat_destino,...,status,local_origem,local_destino,trajeto,hora,dia_util,faixa_distancia_km,velocidade_media_kmh,outlier_tempo,tempo_transito_min_bruto
tipo,str,"datetime64[us, America/Sao_Paulo]",category,category,category,category,str,float64,float64,float64,...,category,str,str,str,int32,bool,category,float64,bool,float64


## 8. Conclusões

**Conclusões:**
- **A regra do notebook 01 estava errada.** A regra "velocidade < 3 km/h" marcou 56 entregas, mas só 24 eram erros: 32 eram entregas curtas e normais (41 dos casos estão na faixa de 0 a 2 km, onde o tempo fixo pesa mais). Ela também deixou passar 8 erros.
- **Regra adotada:** tempo maior que 3 vezes a mediana da faixa de distância. Ela encontrou os 32 erros propositais com só 2 falsos positivos (34 marcados). Os limites de 2,5 (6 falsos positivos) e de 4 (3 erros perdidos) foram piores.
- **Impacto:** o tempo máximo caiu de 297,7 para 128,9 min e o desvio padrão de 16,4 para 12,8. A média caiu 0,7 min, e a mediana praticamente não mudou (de 15,95 para 15,90 min). O OTD subiu de 91,9% para 92,6%, porque os erros apareciam todos como "com atraso".
- **O achado do notebook 01 continua válido:** as entregas entre cidades seguem com OTD bem menor (84,2%) que as entregas dentro da mesma cidade (94,8%).
- **Bases geradas:** `data/interim/entregas_validadas.parquet` e `data/processed/entregas_limpas.parquet`, com 5.000 linhas cada. Nenhuma linha foi apagada.


**Próximos passos:**
1. `03_desempenho`: gráficos de OTD por hora, cidade e trajeto usando a base `processed`.
2. Geocodificar os bairros (coordenadas reais) para mapas e roteirização.